In [ ]:
import os 
import numpy as np
from functools import partial
import matplotlib.pyplot as plt
from PIL import Image 


from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms

import random

# from posteriors import Diffusion_Coefficients, BrownianPosterior_Coefficients, get_time_schedule
from typing import List, Optional, Dict


torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True

toy数据集

In [ ]:
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
import math

class Checkerboard(Dataset):
    def __init__(self, size=8, grid_size=4):
        self.size = size
        self.grid_size = grid_size
        self.checkboard = torch.tensor([[i, j] for i in range(grid_size) for j in range(grid_size) if (i + j) % 2 == 0])

        grid_pos = torch.randint(low=0, high=self.checkboard.shape[0], size=(self.size,), dtype=torch.int64)
        self.data = torch.rand(size=(self.size, 2), dtype=torch.float32) + self.checkboard[grid_pos].float()
        self.data = self.data / self.grid_size * 2 - 1

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        return self.data[idx]


class Pinwheel(Dataset):
    def __init__(self, size: int, num_classes:int=8):
        self.size = size

        radial_std = 0.3
        tangential_std = 0.1
        num_per_class = math.ceil(size / num_classes)
        rate = 0.25
        rads = np.linspace(0, 2 * np.pi, num_classes, endpoint=False)

        features = np.random.randn(num_classes*num_per_class, 2) \
            * np.array([radial_std, tangential_std])
        features[:, 0] += 1.
        labels = np.repeat(np.arange(num_classes), num_per_class)

        angles = rads[labels] + rate * np.exp(features[:, 0])
        rotations = np.stack([np.cos(angles), -np.sin(angles), np.sin(angles), np.cos(angles)])
        rotations = np.reshape(rotations.T, (-1, 2, 2))
        x = .4 * np.random.permutation(np.einsum("ti,tij->tj", features, rotations))

        self.init_sample = torch.from_numpy(x).float()

    def __len__(self):
        return self.size

    def __getitem__(self, idx: int) -> Tensor:
        return self.init_sample[idx]


data_size = 2 ** 15
pinwheel_dataset = Pinwheel(data_size)
checkerboard_dataset = Checkerboard(size=data_size)

In [ ]:
batch_size = 2 ** 10
pinwheel_data_loader = DataLoader(pinwheel_dataset, batch_size, num_workers=0, pin_memory=True, shuffle=True)
checkerboard_data_loader = DataLoader(checkerboard_dataset, batch_size, num_workers=0, pin_memory=True, shuffle=True)

def show_2d_data(data: Tensor):
    plt.figure(figsize=(3, 3))
    plt.scatter(data[:, 0], data[:, 1])
    plt.xlim(-1.1, 1.1)
    plt.ylim(-1.1, 1.1)

    plt.show()
    plt.close()

In [ ]:
pinwheel_batch = next(iter(pinwheel_data_loader))
checkerboard_batch = next(iter(checkerboard_data_loader))

In [ ]:
show_2d_data(next(iter(pinwheel_data_loader)))
show_2d_data(next(iter(checkerboard_data_loader)))

$\varepsilon$ 是布朗桥强度

对于离散的情况

从$x_0$和$x_1$的联合分布中采样$(x_0, x_1)$, 然后得出条件化的布朗桥

$$

p^{W^\varepsilon}(x_{t_1}\ldots x_{t_N} | x_0, x_1) = \prod_{i=1}^{N} p^{W^\varepsilon}(x_{t_i} | x_{t_{i-1}}, x_1) \\

p^{W^\varepsilon}(x_{t_i} | x_{t_{i-1}}, x_1) = \mathcal{N}(x_{t_i}; x_{t_{i-1}} + \frac{t_i - t_{i-1}}{1 - t_{i-1}}(x_1 - x_{t_{i-1}}),\varepsilon \frac{(t_i - t_{i-1})(1 - t_i)}{(1 - t_{i-1})} I)\\

p^{W^\varepsilon}(x_{t_{i}} | x_{t_{i+1}}, x_1) = \mathcal{N}(x_{t_i}; \frac{t_{i+1}-t_{i}}{t_{i+1}} x_0 + \frac{t_{i}}{t_{i+1}} x_{t_{i+1}},\varepsilon \frac{t_i(t_{i+1} - t_{i})}{t_{i+1}} I)\\

p^{W^\varepsilon}(x_{t_N} | x_0, x_1) = \mathcal{N}(x_{t_N}; (1 - t_N) x_0 + t_N x_1, \varepsilon t_N(1 - t_N) I)

$$

MSBM训练:

我们假定有N个均匀时间网格

$$
t_0 =0 \lt t_1 \lt \cdots \lt t_N = 1
$$

然后我们给定$m+1$个边界限定
$$
t_{k_0} = 0 \lt t_{k_1} \lt \cdots \lt t_{k_m} = 1, 
$$
Let $N_j= k_{j+1}-k_j$.

在第$j$段内，我们引入局部归一化时间:
$$
\tau_i = \frac{i}{N_j},\quad i=0,\ldots,N_j.
$$
这样在内部套用ASBM的布朗桥系数公式


In [ ]:
class SegmentPosterior:
    """
    一段 [k_j, k_{j+1}] 内部用到的局部布朗桥系数与便捷采样。
    """
    
    def __init__(self, N_j: int, epsilon: float, device):
        self.N_j = N_j
        self.epsilon = epsilon
        t = torch.linspace(0, 1, N_j+1, device=device)  # 段内局部时间
        self.posterior_mean_coef1 = 1 - t[:-1]/t[1:]    # for x_left
        self.posterior_mean_coef2 = t[:-1]/t[1:]        # for x_{t+1}
        var = epsilon * t[:-1]*(t[1:]-t[:-1]) / t[1:]
        self.posterior_log_variance = torch.log(var.clamp(min=1e-20))
        self.tau_tp1 = (torch.arange(N_j, device=device, dtype=torch.float32)+1.0)/float(N_j)

    def sample_bridge_tp1(self, x_left, x_right, local_i):
        """采样 x_{t+1}（段内第 i->i+1 步的右端）"""
        tau = self.tau_tp1[local_i].unsqueeze(-1)
        mean = (1. - tau) * x_left + tau * x_right
        var = self.epsilon * tau * (1. - tau)
        noise = torch.randn_like(x_left)
        mask = (tau>0) & (tau<1)
        return mean + mask * torch.sqrt(var.clamp_min(1e-20)) * noise

    def sample_posterior_t(self, x_left, x_tp1, local_i):
        """采样 x_t ~ q(x_t | x_{t+1}, x_left)（段内单步后验）"""
        a = self.posterior_mean_coef1[local_i].unsqueeze(-1)  # coef for x_left
        b = self.posterior_mean_coef2[local_i].unsqueeze(-1)  # coef for x_{t+1}
        mean = a * x_left + b * x_tp1
        logv = self.posterior_log_variance[local_i].unsqueeze(-1)
        noise = torch.randn_like(x_tp1)
        nonzero = (local_i > 0).float().unsqueeze(-1)
        return mean + nonzero * torch.exp(0.5*logv) * noise



步数映射器，并预处理系数

In [ ]:
class TimeLayout:
    def __init__(self, N:int, k_list:List[int], epsilon, device):
        assert k_list[0]==0 and k_list[-1]==N
        assert all(k_list[i]<k_list[i+1] for i in range(len(k_list)-1))
        self.N = N
        self.k_list = k_list
        self.epsilon = float(epsilon)
        self.device = device

        self.seg_ranges = [(k_list[j], k_list[j+1]) for j in range(len(k_list)-1)]
        self.seg_N = [b-a for (a,b) in self.seg_ranges]

        # 建立映射：全局 t -> 段 id、段内局部 i
        seg_id = torch.empty(N, dtype=torch.long)
        local_i = torch.empty(N, dtype=torch.long)
        for j,(a,b) in enumerate(self.seg_ranges):
            seg_id[a:b] = j
            local_i[a:b] = torch.arange(0, b-a, dtype=torch.long)
        self.seg_id = seg_id.to(device)
        self.local_i = local_i.to(device)

        # 预计算：把每段的桥系数“展开”为全局步长上的向量
        # posterior 单步：x_t | x_{t+1}, x_left  ~ N(a*x_left + b*x_{t+1},  exp(logv))
        global_a   = torch.empty(N, device=device, dtype=torch.float32)
        global_b   = torch.empty(N, device=device, dtype=torch.float32)
        global_logv= torch.empty(N, device=device, dtype=torch.float32)
        global_tau = torch.empty(N, device=device, dtype=torch.float32)  # τ_{i+1} = (i+1)/N_j

        for j,(a,b) in enumerate(self.seg_ranges):
            Nj = b - a
            # 段内局部时间格
            tloc = torch.linspace(0., 1., Nj+1, device=device)  # [0,1]
            # 对应 i=0..Nj-1 的系数
            a_post = 1. - tloc[:-1]/tloc[1:]     # coef for x_left
            b_post = tloc[:-1]/tloc[1:]          # coef for x_{t+1}
            var_post = self.epsilon * tloc[:-1] * (tloc[1:]-tloc[:-1]) / tloc[1:]  # ε * t_i * Δ / t_{i+1}
            tau_tp1  = (torch.arange(Nj, device=device, dtype=torch.float32)+1.0)/float(Nj)

            sl = slice(a,b)
            global_a[sl]    = a_post
            global_b[sl]    = b_post
            global_logv[sl] = torch.log(var_post.clamp_min(1e-20))
            global_tau[sl]  = tau_tp1

        self.global_a     = global_a            # (N,)
        self.global_b     = global_b            # (N,)
        self.global_logv  = global_logv         # (N,)
        self.global_tau   = global_tau          # (N,)


In [ ]:
class Sampler:
    def __init__(self, layout: TimeLayout, device):
        self.layout = layout
        self.device = device

    def sample_pair_condition(self, anchors_tuple, t_idx: torch.Tensor):
        """
        返回真一步对 (x_t, x_{t+1})，全向量实现：
          1) 先采 x_{t+1} ~ N( (1-τ)x_L + τ x_R,  ε τ(1-τ) I )
          2) 再采 x_t     ~ N( a x_L + b x_{t+1},  Var_post )
        """
        assert t_idx.dtype in (torch.int32, torch.int64), "t_idx must be int tensor"
        N = self.layout.N
        if not torch.all((t_idx >= 0) & (t_idx < N)):
            raise ValueError("t_idx out of valid range [0, N-1]")

        B = t_idx.size(0)
        D = anchors_tuple[0].size(1)

        # 1) 针对每个样本，根据 seg_id 选出对应的 (x_L, x_R)
        # anchors: (m+1, B, D)
        anchors = torch.stack(anchors_tuple, dim=0).to(self.device)  # [J+1, B, D]
        seg = self.layout.seg_id[t_idx]                               # (B,)
        bidx = torch.arange(B, device=self.device)                    # (B,)
        xL = anchors[seg,     bidx, :]                                # (B,D)
        xR = anchors[seg + 1, bidx, :]                                # (B,D)

        # 2) 采 x_{t+1}（桥一步）
        tau = self.layout.global_tau[t_idx].unsqueeze(-1)             # (B,1)
        mean_tp1 = (1. - tau) * xL + tau * xR                         # (B,D)
        var_tp1  = self.layout.epsilon * tau * (1. - tau)             # (B,1)
        x_tp1 = mean_tp1 + torch.sqrt(var_tp1) * torch.randn(B, D, device=self.device)

        # 3) 采 x_t（单步后验）
        a = self.layout.global_a[t_idx].unsqueeze(-1)                 # (B,1)
        b = self.layout.global_b[t_idx].unsqueeze(-1)                 # (B,1)
        mean_t = a * xL + b * x_tp1                                   # (B,D)

        logv  = self.layout.global_logv[t_idx].unsqueeze(-1)          # (B,1)
        std_t = torch.exp(0.5 * logv)

        # i=0 时方差为 0（与原实现一致）
        nonzero = (self.layout.local_i[t_idx] > 0).float().unsqueeze(-1)
        x_t = mean_t + nonzero * std_t * torch.randn(B, D, device=self.device)

        return x_t, x_tp1
    
    def step_std(self, t_idx: torch.Tensor) -> torch.Tensor:
        """
        返回对应一步后验的标准差 σ_i，向量化。
        形状：(B,1)
        """
        assert t_idx.dtype in (torch.int32, torch.int64)
        N = self.layout.N
        if not torch.all((t_idx >= 0) & (t_idx < N)):
            raise ValueError("t_idx out of valid range [0, N-1]")

        std = torch.exp(0.5 * self.layout.global_logv[t_idx]).unsqueeze(-1)  # (B,1)
        return std


接下来需要一个统一的DataLoader，返回多锚点配对数据，作为从联合分布q采样的工具

In [ ]:
class MultiAnchorDataset(Dataset):
    """
    返回一条多锚点样本 (x_k0, x_k1, ..., x_km)。
    默认各锚点独立采样；如需更强耦合可后续替换为 minibatch OT。
    """
    def __init__(self, dist_classes, dist_kwargs_list, size, pairing="independent"):
        assert len(dist_classes) == len(dist_kwargs_list)
        self.dist_classes = dist_classes
        self.dist_kwargs_list = dist_kwargs_list
        self.size = size
        self.pairing = pairing
        self.regenerate()

    def regenerate(self):
        self.datasets = [C(size=self.size, **kw) for C,kw in zip(self.dist_classes, self.dist_kwargs_list)]
        print(f"Regenerated {len(self.datasets)} anchor datasets, size={self.size}")

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        if self.pairing == "aligned":
            ii = [idx % len(ds) for ds in self.datasets]
        else:
            ii = [random.randint(0, len(ds)-1) for ds in self.datasets]
        return tuple(ds[i] for ds,i in zip(self.datasets, ii))

接下来的训练，我们要能做到以下几个能力
1. 给定$x_0, x_1\sim q(x_0,x_1)$ （这里直接假设独立）, 构造 $(x_t, x_{t+1})$ 作为布朗桥ground truth, 先按$q(x_{t+1}| x_0, x_1)$采样$x_{t+1}$，再按$q(x_t|x_{t+1}, x_0)$采样$x_{t}$
2. 生成器：给定$(x_t,t)$， 预测 $x_0$
3. 判别器：给定$(x_t, x_{t+1})$，判断是不是由生成器生成的

现在是判别器和生成器

In [ ]:
class MyGenerator(nn.Module):
    def __init__(
        self, x_dim, t_dim, z_dim, n_t, out_dim, layers,
        active=partial(nn.LeakyReLU, 0.2),
    ):
        super().__init__()

        self.x_dim = x_dim
        self.t_dim = t_dim
        self.z_dim = z_dim

        self.model_list = []
        ch_prev = x_dim + t_dim + z_dim

        self.t_transform = nn.Embedding(n_t, t_dim,)

        for ch_next in layers:
            self.model_list.append(nn.Linear(ch_prev, ch_next))
            self.model_list.append(active())
            ch_prev = ch_next

        self.model_list.append(nn.Linear(ch_prev, out_dim))
        self.model = nn.Sequential(*self.model_list)

    def forward(self, x, t, z):
        batch_size = x.shape[0]

        return self.model(
            torch.cat([
                x,
                self.t_transform(t),
                z,
            ], dim=1)
        )


class MyDiscriminator(nn.Module):
    def __init__(
        self, x_dim, t_dim, n_t, layers,
        active=partial(nn.LeakyReLU, 0.2),
    ):
        super().__init__()

        self.x_dim = x_dim
        self.t_dim = t_dim

        self.model_list = []
        ch_prev = 2 * x_dim + t_dim

        self.t_transform = nn.Embedding(n_t, t_dim,)
        
        for ch_next in layers:
            # print(f"Build layer from {ch_prev} to {ch_next}")
            self.model_list.append(nn.Linear(ch_prev, ch_next))
            self.model_list.append(active())
            ch_prev = ch_next

        self.model_list.append(nn.Linear(ch_prev, 1))
        self.model = nn.Sequential(*self.model_list)

    def forward(self, x_t, t, x_tp1,):
        transform_t = self.t_transform(t)
        # print(f"x_t.shape = {x_t.shape}, transform_t = {transform_t.shape}, x_tp1 = {x_tp1.shape}")

        return self.model(
            torch.cat([
                x_t,
                transform_t,
                x_tp1,
            ], dim=1)
        ).squeeze()

必要的参数
1. Batch size
2. learning rate: lr_d, lr_g
3. optimizer: Adam beta1, beta2
4. epoch, 迭代次数
5. epsilon, 布朗桥强度
6. num_timesteps, 时间步数
7. x_dim, 数据维度
8. t_dim, 时间维度
9. z_dim, 隐空间维度
10. output_dir, 输出目录
11. seed, 随机种子
12. use_minibatch_ot, 是否使用minibatch OT
13. use_r1, 是否使用r1正则化
14. r1_gamma, r1正则化系数
15. lazy_reg, r1正则化的间隔
16. use_ema, 是否使用ema
17. ema_decay, ema衰减系数
18. save_ckpt, 是否保存模型
19. ckpt_interval, 保存模型的间隔
20. print, 是否打印训练情况
21. print_interval, 打印训练情况的间隔
22. vis, 是否可视化
23. vis_interval, 可视化的间隔
24. resume, 是否从断点继续训练

In [ ]:
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def parse_args(usedefault:bool = False):
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--batch_size', type=int, default=2**17)
    parser.add_argument('--lr_d', type=float, default=1e-3)
    parser.add_argument('--lr_g', type=float, default=1e-3)
    parser.add_argument('--beta1', type=float, default=0.5)
    parser.add_argument('--beta2', type=float, default=0.9)
    parser.add_argument('--epoch', type=int, default=100)
    parser.add_argument('--epsilon', type=float, default=0.5)
    parser.add_argument('--x_dim', type=int, default=2)
    parser.add_argument('--t_dim', type=int, default=6)
    parser.add_argument('--z_dim', type=int, default=2)
    parser.add_argument('--num_timesteps', type=int, default=12)
    parser.add_argument('--seed', type=int, default=42)
    parser.add_argument('--output_dir', type=str, default='./output')
    parser.add_argument('--use_minibatch_ot', action='store_true', help='whether to use minibatch OT coupling')
    parser.add_argument('--use_r1', action='store_true', help='whether to use r1 regularization')
    parser.add_argument('--r1_gamma', type=float, default=0.01, help='r1 regularization coefficient')
    parser.add_argument('--lazy_reg', type=int, default=5, help='r1 regularization interval')
    parser.add_argument('--use_ema', action='store_true', help='whether to use ema')
    parser.add_argument('--ema_decay', type=float, default=0.999, help='ema decay coefficient')
    parser.add_argument('--save_ckpt', action='store_true', help='whether to save model checkpoints')
    parser.add_argument('--print', action='store_true', help='whether to print training progress')
    parser.add_argument('--print_interval', type=int, default=100, help='interval for printing training progress')
    parser.add_argument('--vis', action='store_true', help='whether to visualize training progress')
    parser.add_argument('--vis_interval', type=int, default=1000, help='interval for visualizing training progress')
    parser.add_argument('--resume', action='store_true', help='whether to resume training from a checkpoint')

    if usedefault:
        args = parser.parse_args([])
    else:
        args = parser.parse_args()

    args.k_list = [0, 4, 8, args.num_timesteps]

    return args

In [ ]:
def sample_checkerboard_on_gpu(B, grid, device):
    # 预生成偶数格坐标到 GPU（放到函数外预存更省）
    coords = torch.tensor(
        [[i, j] for i in range(grid) for j in range(grid) if (i + j) % 2 == 0],
        device=device, dtype=torch.float32
    )
    idx = torch.randint(0, coords.size(0), (B,), device=device)
    base = coords[idx] + torch.rand(B, 2, device=device)
    return base / grid * 2 - 1

def sample_pinwheel_on_gpu(B, num_classes, device):
    radial_std, tangential_std, rate = 0.3, 0.1, 0.25
    labels = torch.randint(0, num_classes, (B,), device=device)
    rads = 2*math.pi * torch.arange(num_classes, device=device) / num_classes
    features = torch.randn(B, 2, device=device)
    features *= torch.tensor([radial_std, tangential_std], device=device)
    features[:, 0] += 1.
    angles = rads[labels] + rate * torch.exp(features[:, 0])
    rot = torch.stack([torch.cos(angles), -torch.sin(angles),
                       torch.sin(angles),  torch.cos(angles)], dim=-1).view(B, 2, 2)
    x = 0.4 * torch.einsum('bi,bij->bj', features, rot)
    return x

In [ ]:

class SegmentTransporter:
    """
    用单步生成器做“段内/全程”的推送。
    - scope='segment' 时需要 seg_j，按照该段的全局 t 索引推进；
    - scope='full' 时覆盖全局 t∈{0,...,N-1}。
    - direction:
        'bw' 右→左：x_t = x_{t+1} + dx_bw(x_{t+1}, t)   （t 递减）
        'fw' 左→右：x_{t+1} = x_t + dx_fw(x_t, t)       （t 递增）
    """
    def __init__(self, layout: TimeLayout, args, device):
        self.layout = layout
        self.args = args
        self.device = device

    @torch.no_grad()
    def transport(self, netG, x_start: torch.Tensor,
                  direction: str,
                  scope: str,
                  seg_j: int = None,
                  return_traj: bool = False):
        """
        netG: 单步生成器（bw 或 fw 任意一个）
        x_start:
            - direction='bw' 时应传“右端”样本（segment: 该段右锚；full: 最右锚）
            - direction='fw' 时应传“左端”样本（segment: 该段左锚；full: 最左锚）
        direction: 'bw' 或 'fw'
        scope: 'segment' 或 'full'
        seg_j: scope='segment' 时必填
        return_traj: True 则返回 list（含两端），False 返回终点张量
        """
        assert direction in ('bw', 'fw')
        assert scope in ('segment', 'full')
        if scope == 'segment':
            assert seg_j is not None
            kL, kR = self.layout.seg_ranges[seg_j]
            t_range = range(kL, kR) if direction == 'fw' else reversed(range(kL, kR))
        else:
            N = self.layout.N
            t_range = range(0, N) if direction == 'fw' else reversed(range(0, N))

        x_cur = x_start.clone().to(self.device)
        B = x_cur.size(0)
        if return_traj:
            traj = [x_cur.clone()]  # 含起点

        for t in t_range:
            t_vec = torch.full((B,), t, device=self.device, dtype=torch.long)
            z = torch.randn(B, self.args.z_dim, device=self.device)
            dx = netG(x_cur, t_vec, z)
            x_cur = x_cur + dx  # fw: x_{t+1}=x_t+dx；bw: x_t=x_{t+1}+dx
            if return_traj:
                traj.append(x_cur.clone())

        return traj if return_traj else x_cur


class CondSamplerSegment:
    """
    构造“段 j 的端点联合”样本对，用于桥后验采样一步真对。
    - 对于 bw：返回 (x_left_tilde, x_right_real)
    - 对于 fw：返回 (x_left_real, x_right_tilde)
    """
    def __init__(self, seg_j: int, layout: TimeLayout, transporter: SegmentTransporter):
        self.seg_j = seg_j
        self.layout = layout
        self.transporter = transporter

    @torch.no_grad()
    def sample_bw(self, netG_fw, anchors_tuple: tuple, n: int):
        # 训练 bw：保留左端为真，用 fw 合成右端
        x_left_real = anchors_tuple[self.seg_j][:n].to(self.transporter.device)
        if random.random() < 0:
            x_right_tilde = anchors_tuple[self.seg_j + 1][:n].to(self.transporter.device)
        else:
            x_right_tilde = self.transporter.transport(
                netG_fw, x_left_real, direction='fw', scope='segment', seg_j=self.seg_j
            )
        return x_left_real, x_right_tilde

    @torch.no_grad()
    def sample_fw(self, netG_bw, anchors_tuple: tuple, n: int):
        # 训练 fw：保留右端为真，用 bw 合成左端
        x_right_real = anchors_tuple[self.seg_j + 1][:n].to(self.transporter.device)
        if random.random() < 0:
            x_left_tilde = anchors_tuple[self.seg_j][:n].to(self.transporter.device)
        else:
            x_left_tilde = self.transporter.transport(
                netG_bw, x_right_real, direction='bw', scope='segment', seg_j=self.seg_j
            )
        return x_left_tilde, x_right_real

def markov_projection_one_segment(
    direction: str, seg_j: int,
    netG_proj, netG_cond,
    netD,
    optimizerG, optimizerD,
    sampler: Sampler, layout: TimeLayout,
    condsampler: CondSamplerSegment,
    anchors_tuple: tuple,
    args, device,
    inner_iters: int,
):
    """
    在“段 j、给定方向（bw / fw）”上做 inner_iters 次对抗训练（Markov 投影的一步）。
    真对来自：用“端点联合（真实+合成）”+ 段内后验采样；假对来自：G 位移 + D 判别。
    """
    softplus = F.softplus
    n_t_global = layout.N
    kL, kR = layout.seg_ranges[seg_j]
    # 仅从本段的全局 t 里采样
    def sample_t_idx(B):
        return torch.randint(kL, kR, (B,), device=device, dtype=torch.long)

    for it in range(inner_iters):
        # === 准备端点联合（隐式 pool，按需重采） ===
        B = anchors_tuple[0].size(0)
        # 抽一个 mini-batch 尺寸
        bsz = min(args.batch_size, B)
        if direction == 'bw':
            x_left, x_right = condsampler.sample_bw(netG_cond, anchors_tuple, bsz)
        else:  # 'fw'
            x_left, x_right = condsampler.sample_fw(netG_cond, anchors_tuple, bsz)

        # === Discriminator step ===
        for p in netD.parameters():
            p.requires_grad = True
        netD.zero_grad()

        # 采一个本段时间步
        t_idx = sample_t_idx(bsz)

        # 用桥后验采“真一步对”（注意：桥使用 (x_left, x_right)）
        # Sampler.sample_pair_condition 接受 anchors_tuple 和 t_idx：
        # 我们需要把 anchors_tuple 替换成仅该段的 (x_left, x_right)
        # 为保持统一接口，这里构造一份“临时 anchors”，只替换段 j 的左右锚。
        tmp_anchors = list(anchors_tuple)
        tmp_anchors[seg_j] = x_left
        tmp_anchors[seg_j + 1] = x_right
        tmp_anchors = tuple(tmp_anchors)

        x_t_real, x_tp1_real = sampler.sample_pair_condition(tmp_anchors, t_idx)
        
        if args.use_r1 and (it % args.lazy_reg == 0):
            x_t_real.requires_grad = True

        D_real = netD(x_t_real, t_idx, x_tp1_real.detach()).view(-1)
        errD_real = softplus(-D_real).mean()

        # R1（可选）
        if args.use_r1 and (it % args.lazy_reg == 0):
            grad = torch.autograd.grad(outputs=D_real.sum(), inputs=x_t_real, create_graph=True, retain_graph=True, only_inputs=True)[0]
            grad_penalty = (grad.view(grad.size(0), -1).norm(2, dim=1) ** 2).mean()
            (0.5 * args.r1_gamma * grad_penalty).backward(retain_graph=True)

        errD_real.backward()

        # 假对（基于位移参数化）
        z = torch.randn(bsz, args.z_dim, device=device)
        if direction == 'bw':
            with torch.no_grad():
                dx = netG_proj(x_tp1_real, t_idx, z)
                x_t_fake = x_tp1_real + dx
            D_fake = netD(x_t_fake, t_idx, x_tp1_real.detach()).view(-1)
        else:
            with torch.no_grad():
                dx = netG_proj(x_t_real, t_idx, z)
                x_tp1_fake = x_t_real + dx
            D_fake = netD(x_t_real.detach(), t_idx, x_tp1_fake).view(-1)

        errD_fake = softplus(D_fake).mean()
        errD_fake.backward()
        nn.utils.clip_grad_norm_(netD.parameters(), max_norm=1.0)
        optimizerD.step()

        # === Generator step ===
        for p in netD.parameters():
            p.requires_grad = False
        netG_proj.zero_grad()

        # 重新采一批（避免 D/G 信息泄露）
        t_idx = sample_t_idx(bsz)
        x_t_real, x_tp1_real = sampler.sample_pair_condition(tmp_anchors, t_idx)

        z = torch.randn(bsz, args.z_dim, device=device)
        if direction == 'bw':
            x_t_fake = x_tp1_real.detach() + netG_proj(x_tp1_real.detach(), t_idx, z)
            out = netD(x_t_fake, t_idx, x_tp1_real.detach()).view(-1)
        else:
            x_tp1_fake = x_t_real.detach() + netG_proj(x_t_real.detach(), t_idx, z)
            out = netD(x_t_real.detach(), t_idx, x_tp1_fake).view(-1)

        errG = softplus(-out).mean()
        errG.backward()
        nn.utils.clip_grad_norm_(netG_proj.parameters(), max_norm=1.0)
        optimizerG.step()
    


def alternating_train(args):
    """
    外层交替（IMF-style）总控：对每一段进行 bw→fw 的交替训练。
    """
    set_seed(args.seed)
    os.makedirs(args.output_dir, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    N = int(args.num_timesteps)
    layout = TimeLayout(N, args.k_list, args.epsilon, device=device)
    sampler = Sampler(layout=layout, device=device)

    # === 数据 ===
    multi_ds = MultiAnchorDataset(
        dist_classes=[Checkerboard, Checkerboard, Pinwheel, Pinwheel],
        dist_kwargs_list=[{'grid_size':4}, {'grid_size':3}, {'num_classes':4}, {'num_classes':6}],
        size=args.batch_size,
        pairing="independent",
    )
    sb_loader = DataLoader(multi_ds, batch_size=args.batch_size, shuffle=True, drop_last=True)

    # === 模型（双向） ===
    def build_G():
        return MyGenerator(
            x_dim=args.x_dim, t_dim=args.t_dim, z_dim=args.z_dim,
            n_t=args.num_timesteps, out_dim=args.x_dim,
            layers=[128,256,256,128],
        ).to(device)
    def build_D():
        return MyDiscriminator(
            x_dim=args.x_dim, t_dim=args.t_dim, n_t=args.num_timesteps,
            layers=[128,256,256,128],
        ).to(device)

    netG_bw, netD_bw = build_G(), build_D()
    netG_fw, netD_fw = build_G(), build_D()

    optG_bw = optim.Adam(netG_bw.parameters(), lr=args.lr_g, betas=(args.beta1, args.beta2))
    optD_bw = optim.Adam(netD_bw.parameters(), lr=args.lr_d, betas=(args.beta1, args.beta2))
    optG_fw = optim.Adam(netG_fw.parameters(), lr=args.lr_g, betas=(args.beta1, args.beta2))
    optD_fw = optim.Adam(netD_fw.parameters(), lr=args.lr_d, betas=(args.beta1, args.beta2))

    if args.use_ema:
        from torch_ema import ExponentialMovingAverage
        ema_bw = ExponentialMovingAverage(netG_bw.parameters(), decay=args.ema_decay).to(device)
        ema_fw = ExponentialMovingAverage(netG_fw.parameters(), decay=args.ema_decay).to(device)
    else:
        ema_bw = ema_fw = None

    transporter = SegmentTransporter(layout, args, device)
    
    if args.resume:
        ckpt_path = os.path.join(args.output_dir, 'latest_ckpt.pth')
        if os.path.isfile(ckpt_path):
            print(f"Loading checkpoint from {ckpt_path} ...")
            ckpt = torch.load(ckpt_path, map_location=device)
            netG_bw.load_state_dict(ckpt['netG_bw'])
            netD_bw.load_state_dict(ckpt['netD_bw'])
            netG_fw.load_state_dict(ckpt['netG_fw'])
            netD_fw.load_state_dict(ckpt['netD_fw'])
            optG_bw.load_state_dict(ckpt['optG_bw'])
            optD_bw.load_state_dict(ckpt['optD_bw'])
            optG_fw.load_state_dict(ckpt['optG_fw'])
            optD_fw.load_state_dict(ckpt['optD_fw'])
            if args.use_ema and (ema_bw is not None) and ('ema_bw' in ckpt) and ('ema_fw' in ckpt) and (ema_fw is not None):
                ema_bw.load_state_dict(ckpt['ema_bw'])
                ema_fw.load_state_dict(ckpt['ema_fw'])
            print("Checkpoint loaded.")
        else:
            print(f"No checkpoint found at {ckpt_path}, training from scratch.")

    # === 交替外层循环 ===
    num_segments = len(args.k_list) - 1
    print(f"Starting alternating training: segments={num_segments}, outer_iters={args.outer_iters}")
    for outer in range(args.outer_iters):
        print(f"=== Outer iteration {outer+1}/{args.outer_iters} ===")
        for anchors in sb_loader:  # 每个 batch 都用来更新
            anchors = tuple(a.to(device) for a in anchors)  # (B,D) * (m+1)
            B = anchors[0].size(0)
            # print(f"[Outer {outer+1}/{args.outer_iters}] Backward (R→L) per-segment training...")
            # 逐段训练 bw
            for j in range(num_segments):
                condsampler = CondSamplerSegment(j, layout, transporter)
                markov_projection_one_segment(
                    direction='bw', seg_j=j,
                    netG_proj=netG_bw, netG_cond=netG_fw,
                    netD=netD_bw,
                    optimizerG=optG_bw, optimizerD=optD_bw,
                    sampler=sampler, layout=layout,
                    condsampler=condsampler,
                    anchors_tuple=anchors,
                    args=args, device=device,
                    inner_iters=args.inner_iters_per_segment,
                )
            if args.use_ema and (ema_bw is not None): 
                ema_bw.update()

            # print(f"[Outer {outer+1}/{args.outer_iters}] Forward (L→R) per-segment training...")
            # 逐段训练 fw
            for j in range(num_segments):
                condsampler = CondSamplerSegment(j, layout, transporter)
                markov_projection_one_segment(
                    direction='fw', seg_j=j,
                    netG_proj=netG_fw, netG_cond=netG_bw,
                    netD=netD_fw,
                    optimizerG=optG_fw, optimizerD=optD_fw,
                    sampler=sampler, layout=layout,
                    condsampler=condsampler,
                    anchors_tuple=anchors,
                    args=args, device=device,
                    inner_iters=args.inner_iters_per_segment,
                )
            if args.use_ema and (ema_fw is not None): 
                ema_fw.update()

        # （可选）可视化与 checkpoint
        if args.vis and ((outer+1) % args.vis_outer_every == 0):
            with torch.no_grad():
                N = layout.N
                ncols = N + 1
                n_vis = min(1024, B)

                # 取最左/最右锚点的真实样本各一批
                x_left_true  = anchors[0][:n_vis].to(device)                 # t=0 的真实样本
                x_right_true = anchors[len(args.k_list)-1][:n_vis].to(device) # t=1 的真实样本

                # 整程轨迹：右→左 与 左→右
                traj_full_bw = transporter.transport(netG_bw, x_right_true, direction='bw', scope='full', return_traj=True)
                traj_full_fw = transporter.transport(netG_fw, x_left_true,  direction='fw', scope='full', return_traj=True)

                # 画图：两行（bw 与 fw），每行 N+1 列（含两端）
                import matplotlib.pyplot as plt
                fig, axes = plt.subplots(2, ncols, figsize=(2*ncols, 5))

                # 第一行：BW（右→左）。为了列顺序与时间从 0→1 对齐，我们把它反转显示（左到右依次 t=0..t=1）
                # 显示序列：traj_full_bw[::-1]，索引 c 对应 t = c/N
                for c in range(ncols):
                    ax = axes[0, c]
                    pts = traj_full_bw[::-1][c]
                    ax.scatter(pts[:,0].cpu(), pts[:,1].cpu(), s=3)
                    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
                    ax.set_xticks([]); ax.set_yticks([])
                    if c == 0:
                        ax.set_title('BW: t=0')
                    elif c == ncols-1:
                        ax.set_title('BW: t=1')
                    else:
                        ax.set_title(f'BW: t={c}/{N}')

                # 第二行：FW（左→右），天然就是 t=0..t=1
                for c in range(ncols):
                    ax = axes[1, c]
                    pts = traj_full_fw[c]
                    ax.scatter(pts[:,0].cpu(), pts[:,1].cpu(), s=3)
                    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
                    ax.set_xticks([]); ax.set_yticks([])
                    if c == 0:
                        ax.set_title('FW: t=0')
                    elif c == ncols-1:
                        ax.set_title('FW: t=1')
                    else:
                        ax.set_title(f'FW: t={c}/{N}')

                plt.suptitle(f'Outer {outer+1}: Full trajectory (N={N})')
                plt.tight_layout()
                os.makedirs(args.output_dir, exist_ok=True)
                out_png = os.path.join(args.output_dir, f'vis_full_outer{outer+1}.png')
                plt.savefig(out_png, dpi=150)
                plt.show()
                plt.close()
        if args.save_ckpt and ((outer+1) % args.ckpt_outer_every == 0):
            torch.save({
                'outer': outer+1,
                'netG_bw': netG_bw.state_dict(),
                'netD_bw': netD_bw.state_dict(),
                'netG_fw': netG_fw.state_dict(),
                'netD_fw': netD_fw.state_dict(),
                'optG_bw': optG_bw.state_dict(),
                'optD_bw': optD_bw.state_dict(),
                'optG_fw': optG_fw.state_dict(),
                'optD_fw': optD_fw.state_dict(),
            }, os.path.join(args.output_dir, f'ckpt_outer_{outer+1}.pth'))

    print("Alternating training finished.")


args = parse_args(usedefault=True)
# 额外的交替相关超参
args.outer_iters = 1000                       # 外层交替次数（IMF 迭代）
args.inner_iters_per_segment = 2           # 每段每个方向的内层步数
args.vis_outer_every = 2                    # 可视化间隔（按 outer 记）  
args.ckpt_outer_every = 5                   # ckpt 间隔（按 outer 记）

# 保持你原来的常用选项
args.print = True
args.use_ema = False
args.use_r1 = False
args.vis = True
args.save_ckpt = True
args.resume = True

print(args)
alternating_train(args)
